<a href="https://colab.research.google.com/github/Zurard/AI_LAB_ASSiGINMENTS/blob/main/SarLog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [13]:
import os

PROJECT_DIR = "/content/drive/MyDrive/SaRLog"

os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/raw", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data/processed", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/embeddings", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)

print("SaRLog project folder created!")

SaRLog project folder created!


need to inspect the log file

In [14]:
BGL_PATH = "/content/drive/MyDrive/SaRLog/data/raw/BGL_2k.log"

with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for i in range(5):
        print(f"LOG {i+1}:")
        print(next(f))
        print("-" * 80)

LOG 1:
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected

--------------------------------------------------------------------------------
LOG 2:
- 1117838573 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.53.276129 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected

--------------------------------------------------------------------------------
LOG 3:
- 1117838976 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.36.156884 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected

--------------------------------------------------------------------------------
LOG 4:
- 1117838978 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.38.026704 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected

--------------------------------------------------------------------------------
LOG 5:
- 1117842440 2005.06.03 R23-M0-NE-C:J05-U01 2

In [15]:
with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    total_lines = sum(1 for _ in f)

print("Total log lines:", total_lines)

Total log lines: 2000


In [16]:
with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    lines = [next(f) for _ in range(20)]

for i, line in enumerate(lines):
    print(f"{i+1}: {line.strip()}")

1: - 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
2: - 1117838573 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.53.276129 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
3: - 1117838976 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.36.156884 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
4: - 1117838978 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.49.38.026704 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
5: - 1117842440 2005.06.03 R23-M0-NE-C:J05-U01 2005-06-03-16.47.20.730545 R23-M0-NE-C:J05-U01 RAS KERNEL INFO 63543 double-hummer alignment exceptions
6: - 1117842974 2005.06.03 R24-M0-N1-C:J13-U11 2005-06-03-16.56.14.254137 R24-M0-N1-C:J13-U11 RAS KERNEL INFO 162 double-hummer alignment exceptions
7: - 1117843015 2005.06.03 R21-M1-N6-C:J08-U11 2005-06-03-16.56.55.309974 R21-M1-N6-C:J08-U11 RA

In [17]:
from collections import Counter

with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    labels = [line.split()[0] for line in f]

label_counts = Counter(labels)

print("Number of logs:", len(labels))
print("\nLabels:")
for label, count in label_counts.items():
    print(f"{label!r}: {count}")

Number of logs: 2000

Labels:
'-': 1857
'APPREAD': 3
'KERNDTLB': 60
'KERNSTOR': 30
'APPCHILD': 1
'KERNRTSP': 2
'KERNTERM': 7
'APPSEV': 17
'KERNMNTF': 11
'KERNREC': 5
'APPTO': 2
'APPRES': 4
'APPOUT': 1


In [18]:
with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if not line.startswith("-"):
            print(line)
            break

APPREAD 1117869872 2005.06.04 R04-M1-N4-I:J18-U11 2005-06-04-00.24.32.432192 R04-M1-N4-I:J18-U11 RAS APP FATAL ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33569



Lets start preprocessng the data

In [19]:
import re

def clean_log(text):
    # Convert everything to lowercase
    text = text.lower()

    # Remove HTML tags
    text = re.sub(r'<.*?>', ' ', text)

    # Remove numbers
    text = re.sub(r'\d+', ' ', text)

    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

In [20]:
with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if not line.startswith("-"):
            example_log = line.strip()
            break

print("RAW LOG:")
print(example_log)

print("\n" + "=" * 80 + "\n")

print("CLEANED LOG:")
print(clean_log(example_log))

RAW LOG:
APPREAD 1117869872 2005.06.04 R04-M1-N4-I:J18-U11 2005-06-04-00.24.32.432192 R04-M1-N4-I:J18-U11 RAS APP FATAL ciod: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33569


CLEANED LOG:
appread r m n i j u r m n i j u ras app fatal ciod failed to read message prefix on control stream ciostream socket to


In [21]:
def parse_log(line):
    parts = line.strip().split(maxsplit=10)

    label = parts[0]
    message = parts[10]

    # Convert BGL labels to binary labels
    if label == "-":
        binary_label = 0
    else:
        binary_label = 1

    return binary_label, message

In [22]:
label, message = parse_log(example_log)

print("LABEL:", label)
print("MESSAGE:", message)

LABEL: 1
MESSAGE: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33569


In [23]:
with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        if not line.startswith("-"):
            example_log = line.strip()
            break

label, message = parse_log(example_log)

print("LABEL:", label)

print("MESSAGE:", message)

LABEL: 1
MESSAGE: failed to read message prefix on control stream (CioStream socket to 172.16.96.116:33569


In [24]:
import pandas as pd

In [25]:
records = []

with open(BGL_PATH, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()

        if not line:
            continue

        try:
            label, message = parse_log(line)
            cleaned_message = clean_log(message)

            records.append({
                "label": label,
                "message": cleaned_message
            })

        except Exception as e:
            print("Skipped line:", line)
            print("Reason:", e)

df = pd.DataFrame(records)

print("Number of processed logs:", len(df))
print("\nFirst 5 rows:")
print(df.head())

Number of processed logs: 2000

First 5 rows:
   label                             message
0      0        cache parity error corrected
1      0        cache parity error corrected
2      0        cache parity error corrected
3      0        cache parity error corrected
4      0  double hummer alignment exceptions


In [26]:
print(df["label"].value_counts())

label
0    1857
1     143
Name: count, dtype: int64


In [27]:
print("NORMAL EXAMPLES")
print(df[df["label"] == 0].head(5).to_string(index=False))

print("\nANOMALY EXAMPLES")
print(df[df["label"] == 1].head(5).to_string(index=False))

NORMAL EXAMPLES
 label                            message
     0       cache parity error corrected
     0       cache parity error corrected
     0       cache parity error corrected
     0       cache parity error corrected
     0 double hummer alignment exceptions

ANOMALY EXAMPLES
 label                                                             message
     1 failed to read message prefix on control stream ciostream socket to
     1 failed to read message prefix on control stream ciostream socket to
     1                                                 tlb error interrupt
     1                                                 tlb error interrupt
     1                                                 tlb error interrupt


In [28]:
PROCESSED_PATH = "/content/drive/MyDrive/SaRLog/data/processed/bgl_processed.csv"

df.to_csv(PROCESSED_PATH, index=False)

print("Saved to:")
print(PROCESSED_PATH)

Saved to:
/content/drive/MyDrive/SaRLog/data/processed/bgl_processed.csv


trying to read the preprocessed file :
here 0 --->  is a normal analog

*   here 1 ---> is an anamoly log
*   here 0 ---> is an normal log
---

In [29]:
df_check = pd.read_csv(PROCESSED_PATH)

print("Rows:", len(df_check))
print("Columns:", list(df_check.columns))
print("\nLabels:")
print(df_check["label"].value_counts())

Rows: 2000
Columns: ['label', 'message']

Labels:
label
0    1857
1     143
Name: count, dtype: int64


Splitting the data into training and testing

In [30]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Training samples:", len(train_df))
print("Testing samples:", len(test_df))

print("\nTraining labels:")
print(train_df["label"].value_counts())

print("\nTesting labels:")
print(test_df["label"].value_counts())

Training samples: 1600
Testing samples: 400

Training labels:
label
0    1486
1     114
Name: count, dtype: int64

Testing labels:
label
0    371
1     29
Name: count, dtype: int64


In [31]:
TRAIN_PATH = "/content/drive/MyDrive/SaRLog/data/processed/bgl_train.csv"
TEST_PATH = "/content/drive/MyDrive/SaRLog/data/processed/bgl_test.csv"

train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Training and testing datasets saved.")

Training and testing datasets saved.


now its time to convert logs into embeddings and for this we are usign Secure-BERT

In [32]:
!pip install -q transformers sentencepiece

testing if installed or not

In [33]:
import torch

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


checking if transformer is installed or not

In [34]:
import transformers

print("Transformers version:", transformers.__version__)

Transformers version: 5.17.0


now we will Load the Secure-BERT

In [35]:
from transformers import RobertaTokenizer, RobertaModel
import torch

MODEL_NAME = "ehsanaghaei/SecureBERT"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)
model = RobertaModel.from_pretrained(MODEL_NAME)

model = model.to(device)
model.eval()

print("SecureBERT loaded successfully!")

Using device: cuda


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: ehsanaghaei/SecureBERT
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


SecureBERT loaded successfully!


In [36]:
text = df.iloc[0]["message"]

inputs = tokenizer(
    text,
    return_tensors="pt",
    max_length=512,
    padding="max_length",
    truncation=True
)

inputs = {key: value.to(device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

print("Input IDs shape:", inputs["input_ids"].shape)
print("Last hidden state shape:", outputs.last_hidden_state.shape)

Input IDs shape: torch.Size([1, 512])
Last hidden state shape: torch.Size([1, 512, 768])


this is the fucntion to checj how embedding will be like

In [37]:
embedding = outputs.last_hidden_state.mean(dim=1)

print("Embedding shape:", embedding.shape)
print("\nFirst 10 values:")
print(embedding[0, :10])

Embedding shape: torch.Size([1, 768])

First 10 values:
tensor([-0.0125,  0.0182,  0.0045,  0.0206,  0.0351,  0.0106,  0.0098,  0.0246,
         0.0203, -0.0178], device='cuda:0')


In [38]:
# checking if the model in on GPU or not just for safety
print("Model device:", next(model.parameters()).device)
print("Embedding device:", embedding.device)

Model device: cuda:0
Embedding device: cuda:0


genrating embedding for all the 2k logs

In [39]:
def generate_embeddings(texts, batch_size=32):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=512,
            padding=True,
            truncation=True
        )

        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        with torch.no_grad():
            outputs = model(**inputs)

        # Last encoder layer
        token_embeddings = outputs.last_hidden_state

        # Ignore padding tokens during mean pooling
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = token_embeddings * attention_mask

        summed = masked_embeddings.sum(dim=1)
        counts = attention_mask.sum(dim=1)

        embeddings = summed / counts

        all_embeddings.append(embeddings.cpu())

        print(
            f"Processed {min(start + batch_size, len(texts))}/{len(texts)}"
        )

    return torch.cat(all_embeddings, dim=0)

In [40]:
train_texts = train_df["message"].tolist()

train_embeddings = generate_embeddings(
    train_texts,
    batch_size=32
)

print("\nTraining embeddings shape:")
print(train_embeddings.shape)

Processed 32/1600
Processed 64/1600
Processed 96/1600
Processed 128/1600
Processed 160/1600
Processed 192/1600
Processed 224/1600
Processed 256/1600
Processed 288/1600
Processed 320/1600
Processed 352/1600
Processed 384/1600
Processed 416/1600
Processed 448/1600
Processed 480/1600
Processed 512/1600
Processed 544/1600
Processed 576/1600
Processed 608/1600
Processed 640/1600
Processed 672/1600
Processed 704/1600
Processed 736/1600
Processed 768/1600
Processed 800/1600
Processed 832/1600
Processed 864/1600
Processed 896/1600
Processed 928/1600
Processed 960/1600
Processed 992/1600
Processed 1024/1600
Processed 1056/1600
Processed 1088/1600
Processed 1120/1600
Processed 1152/1600
Processed 1184/1600
Processed 1216/1600
Processed 1248/1600
Processed 1280/1600
Processed 1312/1600
Processed 1344/1600
Processed 1376/1600
Processed 1408/1600
Processed 1440/1600
Processed 1472/1600
Processed 1504/1600
Processed 1536/1600
Processed 1568/1600
Processed 1600/1600

Training embeddings shape:
torch.

In [41]:
test_texts = test_df["message"].tolist()

test_embeddings = generate_embeddings(
    test_texts,
    batch_size=32
)

print("\nTest embeddings shape:")
print(test_embeddings.shape)

Processed 32/400
Processed 64/400
Processed 96/400
Processed 128/400
Processed 160/400
Processed 192/400
Processed 224/400
Processed 256/400
Processed 288/400
Processed 320/400
Processed 352/400
Processed 384/400
Processed 400/400

Test embeddings shape:
torch.Size([400, 768])


In [42]:
TRAIN_EMBEDDINGS_PATH = (
    "/content/drive/MyDrive/SaRLog/embeddings/train_embeddings.pt"
)

TEST_EMBEDDINGS_PATH = (
    "/content/drive/MyDrive/SaRLog/embeddings/test_embeddings.pt"
)

torch.save(train_embeddings, TRAIN_EMBEDDINGS_PATH)
torch.save(test_embeddings, TEST_EMBEDDINGS_PATH)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [43]:
TRAIN_LABELS_PATH = (
    "/content/drive/MyDrive/SaRLog/embeddings/train_labels.pt"
)

TEST_LABELS_PATH = (
    "/content/drive/MyDrive/SaRLog/embeddings/test_labels.pt"
)

torch.save(
    torch.tensor(train_df["label"].values),
    TRAIN_LABELS_PATH
)

torch.save(
    torch.tensor(test_df["label"].values),
    TEST_LABELS_PATH
)

print("Labels saved successfully.")

Labels saved successfully.


now we need to build the siamese network

In [44]:
import torch
import torch.nn as nn


class EmbeddingNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(768, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, 128)
        )

    def forward(self, x):
        return self.network(x)

In [45]:
class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = EmbeddingNetwork()

    def forward(self, x1, x2):
        z1 = self.encoder(x1)
        z2 = self.encoder(x2)

        return z1, z2

testing the network

In [46]:
siamese = SiameseNetwork().to(device)

sample1 = train_embeddings[:4].to(device)
sample2 = train_embeddings[4:8].to(device)

z1, z2 = siamese(sample1, sample2)

print("Input shape:", sample1.shape)
print("Output 1 shape:", z1.shape)
print("Output 2 shape:", z2.shape)

Input shape: torch.Size([4, 768])
Output 1 shape: torch.Size([4, 128])
Output 2 shape: torch.Size([4, 128])


creating training pair to check simimlariy

In [47]:
import random

def create_pairs(embeddings, labels, num_pairs=5000):
    pairs_1 = []
    pairs_2 = []
    pair_labels = []

    labels = labels.tolist()

    for _ in range(num_pairs):
        # Randomly choose first sample
        i = random.randrange(len(embeddings))

        # 50% chance: same class
        if random.random() < 0.5:
            same_class_indices = [
                j for j in range(len(embeddings))
                if labels[j] == labels[i] and j != i
            ]

            j = random.choice(same_class_indices)
            pair_label = 0

        # 50% chance: different class
        else:
            different_class_indices = [
                j for j in range(len(embeddings))
                if labels[j] != labels[i]
            ]

            j = random.choice(different_class_indices)
            pair_label = 1

        pairs_1.append(embeddings[i])
        pairs_2.append(embeddings[j])
        pair_labels.append(pair_label)

    return (
        torch.stack(pairs_1),
        torch.stack(pairs_2),
        torch.tensor(pair_labels)
    )

In [48]:
train_labels = torch.tensor(train_df["label"].values)

pair_x1, pair_x2, pair_y = create_pairs(
    train_embeddings,
    train_labels,
    num_pairs=5000
)

print("Pair 1 shape:", pair_x1.shape)
print("Pair 2 shape:", pair_x2.shape)
print("Pair labels shape:", pair_y.shape)

print("\nPair label counts:")
print(torch.bincount(pair_y))

Pair 1 shape: torch.Size([5000, 768])
Pair 2 shape: torch.Size([5000, 768])
Pair labels shape: torch.Size([5000])

Pair label counts:
tensor([2528, 2472])


now we need to bring all the similar ones close and unisimilar ones apart


In [49]:
class ContrastiveLoss(nn.Module):
  # we are using margin == 1 for this as this should be
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, label):
        distance = torch.sqrt(
            torch.sum((z1 - z2) ** 2, dim=1) + 1e-8
        )

        loss = (
            (1 - label) * distance.pow(2)
            +
            label * torch.clamp(
                self.margin - distance,
                min=0.0
            ).pow(2)
        )

        return loss.mean()

In [50]:
# testing the loss before testing
criterion = ContrastiveLoss(margin=1.0)

siamese = SiameseNetwork().to(device)

sample_x1 = pair_x1[:32].to(device)
sample_x2 = pair_x2[:32].to(device)
sample_y = pair_y[:32].float().to(device)

z1, z2 = siamese(sample_x1, sample_x2)

loss = criterion(z1, z2, sample_y)

print("z1 shape:", z1.shape)
print("z2 shape:", z2.shape)
print("Loss:", loss.item())

z1 shape: torch.Size([32, 128])
z2 shape: torch.Size([32, 128])
Loss: 0.5294297933578491


In [51]:
# setting up the training
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(
    pair_x1,
    pair_x2,
    pair_y.float()
)

loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True
)

siamese = SiameseNetwork().to(device)

criterion = ContrastiveLoss(margin=1.0)

optimizer = torch.optim.Adam(
    siamese.parameters(),
    lr=0.001
)

print("Training batches:", len(loader))


Training batches: 157


In [52]:
# training up the data
num_epochs = 50

loss_history = []

for epoch in range(num_epochs):

    siamese.train()

    total_loss = 0.0

    for x1, x2, y in loader:

        x1 = x1.to(device)
        x2 = x2.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        z1, z2 = siamese(x1, x2)

        loss = criterion(z1, z2, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(loader)
    loss_history.append(average_loss)

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss: {average_loss:.4f}"
    )

Epoch [1/50] Loss: 0.0229
Epoch [2/50] Loss: 0.0003
Epoch [3/50] Loss: 0.0001
Epoch [4/50] Loss: 0.0008
Epoch [5/50] Loss: 0.0011
Epoch [6/50] Loss: 0.0001
Epoch [7/50] Loss: 0.0000
Epoch [8/50] Loss: 0.0000
Epoch [9/50] Loss: 0.0000
Epoch [10/50] Loss: 0.0000
Epoch [11/50] Loss: 0.0000
Epoch [12/50] Loss: 0.0000
Epoch [13/50] Loss: 0.0004
Epoch [14/50] Loss: 0.0025
Epoch [15/50] Loss: 0.0001
Epoch [16/50] Loss: 0.0000
Epoch [17/50] Loss: 0.0000
Epoch [18/50] Loss: 0.0000
Epoch [19/50] Loss: 0.0000
Epoch [20/50] Loss: 0.0000
Epoch [21/50] Loss: 0.0000
Epoch [22/50] Loss: 0.0000
Epoch [23/50] Loss: 0.0000
Epoch [24/50] Loss: 0.0000
Epoch [25/50] Loss: 0.0000
Epoch [26/50] Loss: 0.0000
Epoch [27/50] Loss: 0.0000
Epoch [28/50] Loss: 0.0000
Epoch [29/50] Loss: 0.0000
Epoch [30/50] Loss: 0.0000
Epoch [31/50] Loss: 0.0000
Epoch [32/50] Loss: 0.0000
Epoch [33/50] Loss: 0.0000
Epoch [34/50] Loss: 0.0000
Epoch [35/50] Loss: 0.0037
Epoch [36/50] Loss: 0.0001
Epoch [37/50] Loss: 0.0000
Epoch [38/

In [53]:
MODEL_PATH = "/content/drive/MyDrive/SaRLog/results/siamese_model.pt"

torch.save(
    siamese.state_dict(),
    MODEL_PATH
)

print("Model saved to:", MODEL_PATH)

Model saved to: /content/drive/MyDrive/SaRLog/results/siamese_model.pt


In [54]:
siamese.eval()

with torch.no_grad():
    train_z = siamese.encoder(
        train_embeddings.to(device)
    )

    test_z = siamese.encoder(
        test_embeddings.to(device)
    )

print("Train representation shape:", train_z.shape)
print("Test representation shape:", test_z.shape)

Train representation shape: torch.Size([1600, 128])
Test representation shape: torch.Size([400, 128])


Now we have a concrete classifier which we can use to train and test the log

NOW WE ARE BUILDING THE TWO CENTROIDS

In [55]:
train_labels = torch.tensor(
    train_df["label"].values,
    dtype=torch.long
).to(device)

test_labels = torch.tensor(
    test_df["label"].values,
    dtype=torch.long
).to(device)

normal_centroid = train_z[train_labels == 0].mean(dim=0)
anomaly_centroid = train_z[train_labels == 1].mean(dim=0)

print("Normal centroid shape:", normal_centroid.shape)
print("Anomaly centroid shape:", anomaly_centroid.shape)

Normal centroid shape: torch.Size([128])
Anomaly centroid shape: torch.Size([128])


In [56]:
# CHECKING THE TEST LOGS
normal_distance = torch.norm(
    test_z - normal_centroid,
    dim=1
)

anomaly_distance = torch.norm(
    test_z - anomaly_centroid,
    dim=1
)

predictions = (
    anomaly_distance < normal_distance
).long()

print("Predictions:", predictions.shape)
print("\nFirst 20 predictions:")
print(predictions[:20].cpu())

Predictions: torch.Size([400])

First 20 predictions:
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0])


NOW WE CALCULATE AGAINST THE ACTUAL METRICS

In [57]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

y_true = test_labels.cpu().numpy()
y_pred = predictions.cpu().numpy()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.9925
Precision: 0.9643
Recall   : 0.931
F1-score : 0.9474

Confusion Matrix:
[[370   1]
 [  2  27]]


In [58]:
# finding the reference point
# Find the indices of normal training samples
normal_indices = torch.where(train_labels == 0)[0]

# Pick the first normal sample as our reference
reference_index = normal_indices[0]

# Get its 128-dimensional representation
normal_reference = train_z[reference_index]

print("Reference index:", reference_index.item())
print("Reference label:", train_labels[reference_index].item())
print("Reference shape:", normal_reference.shape)

Reference index: 0
Reference label: 0
Reference shape: torch.Size([128])


In [59]:
# Calculate Euclidean distance from every test sample
# to our normal reference sample.

distances = torch.norm(
    test_z - normal_reference,
    dim=1
)

print("Number of test samples:", len(distances))
print("Minimum distance:", distances.min().item())
print("Maximum distance:", distances.max().item())
print("Mean distance:", distances.mean().item())
print("Median distance:", distances.median().item())

Number of test samples: 400
Minimum distance: 3.430836770235146e-08
Maximum distance: 1.0477808713912964
Mean distance: 0.07313255965709686
Median distance: 0.0005054682260379195


In [60]:
normal_test_distances = distances[test_labels == 0]
anomaly_test_distances = distances[test_labels == 1]

print("NORMAL TEST LOGS")
print("Count :", len(normal_test_distances))
print("Mean  :", normal_test_distances.mean().item())
print("Median:", normal_test_distances.median().item())
print("Max   :", normal_test_distances.max().item())

print("\nANOMALY TEST LOGS")
print("Count :", len(anomaly_test_distances))
print("Mean  :", anomaly_test_distances.mean().item())
print("Median:", anomaly_test_distances.median().item())
print("Min   :", anomaly_test_distances.min().item())

NORMAL TEST LOGS
Count : 371
Mean  : 0.002692463109269738
Median: 0.0005054647335782647
Max   : 0.6634422540664673

ANOMALY TEST LOGS
Count : 29
Mean  : 0.9742799997329712
Median: 1.0408508777618408
Min   : 0.001040140283294022


In [61]:
margin = 1.0

paper_predictions = (distances > margin).long()

y_true = test_labels.cpu().numpy()
y_pred = paper_predictions.cpu().numpy()

In [62]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)

print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1-score :", round(f1, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy : 0.9925
Precision: 1.0
Recall   : 0.8966
F1-score : 0.9455

Confusion Matrix:
[[371   0]
 [  3  26]]


now we are done checking the model now we are checking the bigger dateset

In [63]:
import os

raw_path = "/content/drive/MyDrive/SaRLog/data/raw"

print(os.listdir(raw_path))

['BGL_2k.log', '.ipynb_checkpoints', 'BGL.log', 'BGL_100k.log']


In [64]:
full_bgl_path = "/content/drive/MyDrive/SaRLog/data/raw/BGL.log"

with open(full_bgl_path, "r") as f:
    for i in range(5):
        print(f.readline().strip())

- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.363779 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.527847 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.675872 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.823719 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected
- 1117838570 2005.06.03 R02-M1-N0-C:J12-U11 2005-06-03-15.42.50.982731 R02-M1-N0-C:J12-U11 RAS KERNEL INFO instruction cache parity error corrected


In [65]:
import random

full_bgl_path = "/content/drive/MyDrive/SaRLog/data/raw/BGL.log"
bgl_100k_path = "/content/drive/MyDrive/SaRLog/data/raw/BGL_100k.log"

TARGET_NORMAL = 92850
TARGET_ANOMALY = 7150

normal_logs = []
anomaly_logs = []

with open(full_bgl_path, "r") as f:
    for line in f:
        if not line.strip():
            continue

        # First field is the label
        label = line.split(maxsplit=1)[0]

        if label == "-" and len(normal_logs) < TARGET_NORMAL:
            normal_logs.append(line)

        elif label != "-" and len(anomaly_logs) < TARGET_ANOMALY:
            anomaly_logs.append(line)

        # Stop once we have enough of both classes
        if (
            len(normal_logs) == TARGET_NORMAL
            and len(anomaly_logs) == TARGET_ANOMALY
        ):
            break

print("Normal logs collected :", len(normal_logs))
print("Anomaly logs collected:", len(anomaly_logs))

# Combine and shuffle
selected_logs = normal_logs + anomaly_logs

random.seed(42)
random.shuffle(selected_logs)

# Save
with open(bgl_100k_path, "w") as f:
    f.writelines(selected_logs)

print("Total logs saved:", len(selected_logs))
print("Saved to:", bgl_100k_path)

Normal logs collected : 92850
Anomaly logs collected: 7150
Total logs saved: 100000
Saved to: /content/drive/MyDrive/SaRLog/data/raw/BGL_100k.log


Now we have created the 100K logs dataset with around 92K normal logs and rest as anomoly

In [66]:
from collections import Counter

bgl_100k_path = "/content/drive/MyDrive/SaRLog/data/raw/BGL_100k.log"

label_counts = Counter()

with open(bgl_100k_path, "r") as f:
    for line in f:
        if line.strip():
            label = line.split(maxsplit=1)[0]
            label_counts[label] += 1

print("Total logs:", sum(label_counts.values()))
print("\nLabel counts:")
print(label_counts)

print("\nNormal logs:", label_counts["-"])
print("Anomaly logs:", sum(
    count for label, count in label_counts.items()
    if label != "-"
))

Total logs: 100000

Label counts:
Counter({'-': 92850, 'KERNDTLB': 4851, 'APPREAD': 2164, 'KERNRTSP': 127, 'KERNMC': 8})

Normal logs: 92850
Anomaly logs: 7150


Now we need to parse and clean the dataset of 100K logs

In [67]:
import pandas as pd
import re

def parse_log(line):
    parts = line.strip().split(maxsplit=9)

    label = parts[0]
    message = parts[9]

    # Convert BGL labels to binary labels
    if label == "-":
        binary_label = 0       # normal
    else:
        binary_label = 1       # anomaly

    return binary_label, message


def clean_log(text):
    text = text.lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    return text.strip()


data = []

with open(bgl_100k_path, "r") as f:
    for line in f:
        if not line.strip():
            continue

        label, message = parse_log(line)
        cleaned_message = clean_log(message)

        data.append({
            "label": label,
            "message": cleaned_message
        })

df_100k = pd.DataFrame(data)

print("Total logs:", len(df_100k))
print("\nLabel distribution:")
print(df_100k["label"].value_counts())

print("\nFirst 5 rows:")
display(df_100k.head())

Total logs: 100000

Label distribution:
label
0    92850
1     7150
Name: count, dtype: int64

First 5 rows:


,label,message
0,0,instruction cache parity error corrected
1,1,data tlb error interrupt
2,0,generating core
3,0,generating core
4,0,generating core


Training samples: 80000
Test samples: 20000

Training distribution:
label
0    74280
1     5720
Name: count, dtype: int64

Test distribution:
label
0    18570
1     1430
Name: count, dtype: int64


this is the exact split we wanted


```
#                 100,000 logs
#                      │
#              stratified split
#               ┌──────┴──────┐
#               ↓             ↓
#           80,000          20,000
#          TRAIN            TEST
#               │             │
#       74,280 normal    18,570 normal
#        5,720 anomaly     1,430 anomaly
```


                

In [69]:
train_path_100k = "/content/drive/MyDrive/SaRLog/data/processed/bgl_100k_train.csv"
test_path_100k = "/content/drive/MyDrive/SaRLog/data/processed/bgl_100k_test.csv"

train_df_100k.to_csv(train_path_100k, index=False)
test_df_100k.to_csv(test_path_100k, index=False)

print("Saved training set:", train_path_100k)
print("Saved test set:", test_path_100k)

Saved training set: /content/drive/MyDrive/SaRLog/data/processed/bgl_100k_train.csv
Saved test set: /content/drive/MyDrive/SaRLog/data/processed/bgl_100k_test.csv


In [70]:
# checking if secureBert is loaded
print("Device:", device)
print("Model:", model.__class__.__name__)
print("Tokenizer:", tokenizer.__class__.__name__)

Device: cuda
Model: RobertaModel
Tokenizer: RobertaTokenizer


Now we need to create chekpoints embeddings

In [71]:
import os
import torch

def generate_embeddings_checkpointed(
    texts,
    save_path,
    batch_size=32,
    checkpoint_every=5000
):
    all_embeddings = []

    total = len(texts)

    for start in range(0, total, batch_size):

        batch = texts[start:start + batch_size]

        # Tokenize the batch
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            max_length=512,
            padding=True,
            truncation=True
        )

        # Move tokenized inputs to the T4 GPU
        inputs = {
            key: value.to(device)
            for key, value in inputs.items()
        }

        # We don't need gradients because SecureBERT
        # is being used only to create embeddings.
        with torch.no_grad():
            outputs = model(**inputs)

        # Shape:
        # [batch_size, sequence_length, 768]
        token_embeddings = outputs.last_hidden_state

        # Ignore padding tokens when calculating the mean
        attention_mask = inputs["attention_mask"].unsqueeze(-1)

        masked_embeddings = token_embeddings * attention_mask

        summed = masked_embeddings.sum(dim=1)
        counts = attention_mask.sum(dim=1)

        # One 768-D vector per log
        embeddings = summed / counts

        all_embeddings.append(embeddings.cpu())

        processed = min(start + batch_size, total)

        if processed % checkpoint_every < batch_size or processed == total:
            current_embeddings = torch.cat(all_embeddings, dim=0)

            torch.save(
                current_embeddings,
                save_path
            )

            print(
                f"Saved checkpoint: "
                f"{processed}/{total} "
                f"({processed / total * 100:.1f}%)"
            )

    return torch.cat(all_embeddings, dim=0)

Now i need to get the train embeddings

In [72]:
train_embeddings_100k = generate_embeddings_checkpointed(
    train_df_100k["message"].tolist(),
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_train_embeddings.pt"
)

print("\nFinal shape:", train_embeddings_100k.shape)

Saved checkpoint: 5024/80000 (6.3%)
Saved checkpoint: 10016/80000 (12.5%)
Saved checkpoint: 15008/80000 (18.8%)
Saved checkpoint: 20000/80000 (25.0%)
Saved checkpoint: 25024/80000 (31.3%)
Saved checkpoint: 30016/80000 (37.5%)
Saved checkpoint: 35008/80000 (43.8%)
Saved checkpoint: 40000/80000 (50.0%)
Saved checkpoint: 45024/80000 (56.3%)
Saved checkpoint: 50016/80000 (62.5%)
Saved checkpoint: 55008/80000 (68.8%)
Saved checkpoint: 60000/80000 (75.0%)
Saved checkpoint: 65024/80000 (81.3%)
Saved checkpoint: 70016/80000 (87.5%)
Saved checkpoint: 75008/80000 (93.8%)
Saved checkpoint: 80000/80000 (100.0%)

Final shape: torch.Size([80000, 768])


Now i need to generate all the test embeddings

In [73]:
test_embeddings_100k = generate_embeddings_checkpointed(
    test_df_100k["message"].tolist(),
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_test_embeddings.pt"
)

print("\nFinal shape:", test_embeddings_100k.shape)

Saved checkpoint: 5024/20000 (25.1%)
Saved checkpoint: 10016/20000 (50.1%)
Saved checkpoint: 15008/20000 (75.0%)
Saved checkpoint: 20000/20000 (100.0%)

Final shape: torch.Size([20000, 768])


In [74]:
train_labels_100k = torch.tensor(
    train_df_100k["label"].values,
    dtype=torch.long
)

test_labels_100k = torch.tensor(
    test_df_100k["label"].values,
    dtype=torch.long
)

torch.save(
    train_labels_100k,
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_train_labels.pt"
)

torch.save(
    test_labels_100k,
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_test_labels.pt"
)

print("Train labels:", train_labels_100k.shape)
print("Test labels :", test_labels_100k.shape)

print("\nTrain distribution:")
print(torch.bincount(train_labels_100k))

print("\nTest distribution:")
print(torch.bincount(test_labels_100k))

Train labels: torch.Size([80000])
Test labels : torch.Size([20000])

Train distribution:
tensor([74280,  5720])

Test distribution:
tensor([18570,  1430])


In [75]:
# creating the siamest network for this
import torch
import torch.nn as nn

class EmbeddingNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(768, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        # One network is shared by both branches
        self.encoder = EmbeddingNetwork()

    def forward(self, x1, x2):

        # Both inputs pass through the SAME weights
        z1 = self.encoder(x1)
        z2 = self.encoder(x2)

        return z1, z2

In [76]:
model_100k = SiameseNetwork()

print(model_100k)

x = torch.randn(4, 768)

z1, z2 = model_100k(x, x)

print("\nInput shape :", x.shape)
print("Output 1   :", z1.shape)
print("Output 2   :", z2.shape)

SiameseNetwork(
  (encoder): EmbeddingNetwork(
    (network): Sequential(
      (0): Linear(in_features=768, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=32, bias=True)
      (3): ReLU()
      (4): Linear(in_features=32, out_features=1, bias=True)
    )
  )
)

Input shape : torch.Size([4, 768])
Output 1   : torch.Size([4, 1])
Output 2   : torch.Size([4, 1])


In [77]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, y):

        # Euclidean distance between the two outputs
        distance = torch.sqrt(
            torch.sum((z1 - z2) ** 2, dim=1) + 1e-8
        )

        # Paper's contrastive loss
        loss = (
            (1 - y) * 0.5 * distance.pow(2)
            +
            y * 0.5 * torch.clamp(
                self.margin - distance,
                min=0.0
            ).pow(2)
        )

        return loss.mean()

In [78]:
criterion_100k = ContrastiveLoss(margin=1.0)

print("Margin:", criterion_100k.margin)

Margin: 1.0


In [79]:
# Get indices for each class
normal_indices = torch.where(train_labels_100k == 0)[0]
anomaly_indices = torch.where(train_labels_100k == 1)[0]

print("Normal training samples :", len(normal_indices))
print("Anomaly training samples:", len(anomaly_indices))

Normal training samples : 74280
Anomaly training samples: 5720


In [80]:
import random

NUM_PAIRS = 100000
NUM_SIMILAR = NUM_PAIRS // 2
NUM_DISSIMILAR = NUM_PAIRS // 2

pairs_1 = []
pairs_2 = []
pair_labels = []

# -----------------------------
# 1. Similar pairs
# -----------------------------
# y = 1 means the two logs are similar

for _ in range(NUM_SIMILAR):

    # Randomly choose which class to use
    if random.random() < 0.5:
        indices = normal_indices
    else:
        indices = anomaly_indices

    # Pick two different logs from that class
    i, j = random.sample(indices.tolist(), 2)

    pairs_1.append(train_embeddings_100k[i])
    pairs_2.append(train_embeddings_100k[j])

    pair_labels.append(1)


# -----------------------------
# 2. Dissimilar pairs
# -----------------------------
# y = 0 means the two logs are dissimilar

for _ in range(NUM_DISSIMILAR):

    # One normal + one anomaly
    i = random.choice(normal_indices.tolist())
    j = random.choice(anomaly_indices.tolist())

    pairs_1.append(train_embeddings_100k[i])
    pairs_2.append(train_embeddings_100k[j])

    pair_labels.append(0)


# Convert everything to tensors
pairs_1 = torch.stack(pairs_1)
pairs_2 = torch.stack(pairs_2)
pair_labels = torch.tensor(pair_labels, dtype=torch.float32)

print("Pairs 1 shape:", pairs_1.shape)
print("Pairs 2 shape:", pairs_2.shape)
print("Labels shape  :", pair_labels.shape)

print("\nPair label distribution:")
print(torch.bincount(pair_labels.long()))

Pairs 1 shape: torch.Size([100000, 768])
Pairs 2 shape: torch.Size([100000, 768])
Labels shape  : torch.Size([100000])

Pair label distribution:
tensor([50000, 50000])


instead of feeding the entire 100K logs to the model we  divide it onto 32 batches

In [81]:
from torch.utils.data import TensorDataset, DataLoader

pair_dataset = TensorDataset(
    pairs_1,
    pairs_2,
    pair_labels
)

pair_loader = DataLoader(
    pair_dataset,
    batch_size=32,
    shuffle=True
)

print("Number of batches per epoch:", len(pair_loader))

Number of batches per epoch: 3125


now we traint the network

In [82]:
model_100k = SiameseNetwork().to(device)

optimizer = torch.optim.Adam(
    model_100k.parameters(),
    lr=0.001
)

num_epochs = 50

loss_history = []

model_save_path = (
    "/content/drive/MyDrive/SaRLog/results/"
    "siamese_100k_model.pt"
)

for epoch in range(num_epochs):

    model_100k.train()

    total_loss = 0.0

    for x1, x2, y in pair_loader:

        # Move the batch to the T4 GPU
        x1 = x1.to(device)
        x2 = x2.to(device)
        y = y.to(device)

        # Clear gradients from the previous update
        optimizer.zero_grad()

        # Siamese forward pass
        z1, z2 = model_100k(x1, x2)

        # Calculate contrastive loss
        loss = criterion_100k(z1, z2, y)

        # Calculate gradients
        loss.backward()

        # Update network weights
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(pair_loader)

    loss_history.append(average_loss)

    # Save checkpoint after every epoch
    torch.save(
        {
            "epoch": epoch + 1,
            "model_state_dict": model_100k.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": average_loss
        },
        model_save_path
    )

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs} "
        f"- Loss: {average_loss:.6f}"
    )

Epoch 01/50 - Loss: 0.206032
Epoch 02/50 - Loss: 0.205624
Epoch 03/50 - Loss: 0.205553
Epoch 04/50 - Loss: 0.205486
Epoch 05/50 - Loss: 0.205462
Epoch 06/50 - Loss: 0.205443
Epoch 07/50 - Loss: 0.205395
Epoch 08/50 - Loss: 0.205380
Epoch 09/50 - Loss: 0.205364
Epoch 10/50 - Loss: 0.205339
Epoch 11/50 - Loss: 0.205297
Epoch 12/50 - Loss: 0.205294
Epoch 13/50 - Loss: 0.205282
Epoch 14/50 - Loss: 0.205260
Epoch 15/50 - Loss: 0.205259
Epoch 16/50 - Loss: 0.205242
Epoch 17/50 - Loss: 0.205220
Epoch 18/50 - Loss: 0.205221
Epoch 19/50 - Loss: 0.205203
Epoch 20/50 - Loss: 0.205202
Epoch 21/50 - Loss: 0.205166
Epoch 22/50 - Loss: 0.205148
Epoch 23/50 - Loss: 0.205195
Epoch 24/50 - Loss: 0.205130
Epoch 25/50 - Loss: 0.205148
Epoch 26/50 - Loss: 0.205120
Epoch 27/50 - Loss: 0.205117
Epoch 28/50 - Loss: 0.205173
Epoch 29/50 - Loss: 0.205115
Epoch 30/50 - Loss: 0.205109
Epoch 31/50 - Loss: 0.205119
Epoch 32/50 - Loss: 0.205115
Epoch 33/50 - Loss: 0.205085
Epoch 34/50 - Loss: 0.205098
Epoch 35/50 - 

the model stopped learning and now i need to check why is that happening

In [83]:
model_100k.eval()

with torch.no_grad():
    z1_all, z2_all = model_100k(
        pairs_1.to(device),
        pairs_2.to(device)
    )

    pair_distances = torch.abs(z1_all - z2_all).squeeze(1)

similar_distances = pair_distances[pair_labels == 1]
dissimilar_distances = pair_distances[pair_labels == 0]

print("SIMILAR PAIRS")
print("Mean distance  :", similar_distances.mean().item())
print("Median distance:", similar_distances.median().item())
print("Min distance   :", similar_distances.min().item())
print("Max distance   :", similar_distances.max().item())

print("\nDISSIMILAR PAIRS")
print("Mean distance  :", dissimilar_distances.mean().item())
print("Median distance:", dissimilar_distances.median().item())
print("Min distance   :", dissimilar_distances.min().item())
print("Max distance   :", dissimilar_distances.max().item())

SIMILAR PAIRS
Mean distance  : 0.17180730402469635
Median distance: 6.705522537231445e-08
Min distance   : 0.0
Max distance   : 1.1578545570373535

DISSIMILAR PAIRS
Mean distance  : 0.21685025095939636
Median distance: 0.07696804404258728
Min distance   : 0.0011409223079681396
Max distance   : 0.9615644812583923


In [84]:
print("OUTPUT STATISTICS")

print("z1 mean:", z1_all.mean().item())
print("z1 std :", z1_all.std().item())
print("z1 min :", z1_all.min().item())
print("z1 max :", z1_all.max().item())

print("\nz2 mean:", z2_all.mean().item())
print("z2 std :", z2_all.std().item())
print("z2 min :", z2_all.min().item())
print("z2 max :", z2_all.max().item())

OUTPUT STATISTICS
z1 mean: -0.10420767217874527
z1 std : 0.1892661601305008
z1 min : -1.0278098583221436
z1 max : 0.048632390797138214

z2 mean: -0.10575098544359207
z2 std : 0.21362660825252533
z2 min : -1.1861903667449951
z2 max : 0.048632390797138214


we are trying to bring similar one close & dissimilar one far

In [85]:
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, y):

        # Distance between the two Siamese outputs
        distance = torch.sqrt(
            torch.sum((z1 - z2) ** 2, dim=1) + 1e-8
        )

        # y = 1 -> similar
        # y = 0 -> dissimilar
        loss = (
            y * 0.5 * distance.pow(2)
            + (1 - y) * 0.5 *
              torch.clamp(self.margin - distance, min=0.0).pow(2)
        )

        return loss.mean()

genrating the new 100K pairs

In [86]:
normal_indices = torch.where(train_labels_100k == 0)[0]
anomaly_indices = torch.where(train_labels_100k == 1)[0]

print("Normal samples:", len(normal_indices))
print("Anomaly samples:", len(anomaly_indices))

Normal samples: 74280
Anomaly samples: 5720


In [87]:
num_pairs = 100000
half = num_pairs // 2

# --------------------------------------------------
# Similar pairs: normal + normal
# --------------------------------------------------

idx1_same = normal_indices[
    torch.randint(0, len(normal_indices), (half,))
]

idx2_same = normal_indices[
    torch.randint(0, len(normal_indices), (half,))
]

pairs_1_same = train_embeddings_100k[idx1_same]
pairs_2_same = train_embeddings_100k[idx2_same]

labels_same = torch.ones(half)


# --------------------------------------------------
# Dissimilar pairs: normal + anomaly
# --------------------------------------------------

idx_normal = normal_indices[
    torch.randint(0, len(normal_indices), (half,))
]

idx_anomaly = anomaly_indices[
    torch.randint(0, len(anomaly_indices), (half,))
]

pairs_1_diff = train_embeddings_100k[idx_normal]
pairs_2_diff = train_embeddings_100k[idx_anomaly]

labels_diff = torch.zeros(half)


# --------------------------------------------------
# Combine everything
# --------------------------------------------------

pairs_1 = torch.cat([pairs_1_same, pairs_1_diff], dim=0)
pairs_2 = torch.cat([pairs_2_same, pairs_2_diff], dim=0)
pair_labels = torch.cat([labels_same, labels_diff], dim=0)

print("pairs_1:", pairs_1.shape)
print("pairs_2:", pairs_2.shape)
print("labels:", pair_labels.shape)
print("Label distribution:", torch.bincount(pair_labels.long()))

pairs_1: torch.Size([100000, 768])
pairs_2: torch.Size([100000, 768])
labels: torch.Size([100000])
Label distribution: tensor([50000, 50000])


In [88]:
from torch.utils.data import TensorDataset, DataLoader

pair_dataset = TensorDataset(
    pairs_1,
    pairs_2,
    pair_labels
)

pair_loader = DataLoader(
    pair_dataset,
    batch_size=32,
    shuffle=True
)

print("Number of batches:", len(pair_loader))

Number of batches: 3125


creating a new siamese network

In [89]:
class EmbeddingNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(768, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)


class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = EmbeddingNetwork()

    def forward(self, x1, x2):
        z1 = self.encoder(x1)
        z2 = self.encoder(x2)

        return z1, z2

In [90]:
model_100k = SiameseNetwork().to(device)

criterion_100k = ContrastiveLoss(margin=1.0)

optimizer = torch.optim.Adam(
    model_100k.parameters(),
    lr=0.001
)

training it again

In [91]:
num_epochs = 50

loss_history = []

for epoch in range(num_epochs):

    model_100k.train()

    running_loss = 0.0

    for x1, x2, y in pair_loader:

        x1 = x1.to(device)
        x2 = x2.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        z1, z2 = model_100k(x1, x2)

        loss = criterion_100k(z1, z2, y)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(pair_loader)

    loss_history.append(epoch_loss)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} "
        f"- Loss: {epoch_loss:.6f}"
    )

Epoch 01/50 - Loss: 0.001564
Epoch 02/50 - Loss: 0.000025
Epoch 03/50 - Loss: 0.000023
Epoch 04/50 - Loss: 0.000011
Epoch 05/50 - Loss: 0.000004
Epoch 06/50 - Loss: 0.000022
Epoch 07/50 - Loss: 0.000001
Epoch 08/50 - Loss: 0.000000
Epoch 09/50 - Loss: 0.000000
Epoch 10/50 - Loss: 0.000000
Epoch 11/50 - Loss: 0.000000
Epoch 12/50 - Loss: 0.000004
Epoch 13/50 - Loss: 0.000000
Epoch 14/50 - Loss: 0.000000
Epoch 15/50 - Loss: 0.000000
Epoch 16/50 - Loss: 0.000000
Epoch 17/50 - Loss: 0.000000
Epoch 18/50 - Loss: 0.000000
Epoch 19/50 - Loss: 0.000010
Epoch 20/50 - Loss: 0.000000
Epoch 21/50 - Loss: 0.000000
Epoch 22/50 - Loss: 0.000000
Epoch 23/50 - Loss: 0.000000
Epoch 24/50 - Loss: 0.000000
Epoch 25/50 - Loss: 0.000000
Epoch 26/50 - Loss: 0.000000
Epoch 27/50 - Loss: 0.000012
Epoch 28/50 - Loss: 0.000000
Epoch 29/50 - Loss: 0.000000
Epoch 30/50 - Loss: 0.000000
Epoch 31/50 - Loss: 0.000000
Epoch 32/50 - Loss: 0.000000
Epoch 33/50 - Loss: 0.000000
Epoch 34/50 - Loss: 0.000000
Epoch 35/50 - 

In [92]:
model_100k.eval()

with torch.no_grad():
    z1_all, z2_all = model_100k(
        pairs_1.to(device),
        pairs_2.to(device)
    )

# Since each branch outputs a single value,
# Euclidean distance = absolute difference
distances = torch.abs(z1_all - z2_all).squeeze(1)

similar_distances = distances[pair_labels == 1]
dissimilar_distances = distances[pair_labels == 0]

print("===== SIMILAR PAIRS =====")
print("Count :", len(similar_distances))
print("Mean  :", similar_distances.mean().item())
print("Median:", similar_distances.median().item())
print("Min   :", similar_distances.min().item())
print("Max   :", similar_distances.max().item())

print("\n===== DISSIMILAR PAIRS =====")
print("Count :", len(dissimilar_distances))
print("Mean  :", dissimilar_distances.mean().item())
print("Median:", dissimilar_distances.median().item())
print("Min   :", dissimilar_distances.min().item())
print("Max   :", dissimilar_distances.max().item())

===== SIMILAR PAIRS =====
Count : 50000
Mean  : 9.796589495181252e-08
Median: 4.470348358154297e-08
Min   : 0.0
Max   : 2.4437904357910156e-06

===== DISSIMILAR PAIRS =====
Count : 50000
Mean  : 2.0696496963500977
Median: 2.016653537750244
Min   : 1.1289396286010742
Max   : 2.1887218952178955


In [93]:
print("===== SIAMESE OUTPUT STATISTICS =====")

print("z1 mean :", z1_all.mean().item())
print("z1 std  :", z1_all.std().item())
print("z1 min  :", z1_all.min().item())
print("z1 max  :", z1_all.max().item())

print("\nz2 mean :", z2_all.mean().item())
print("z2 std  :", z2_all.std().item())
print("z2 min  :", z2_all.min().item())
print("z2 max  :", z2_all.max().item())

===== SIAMESE OUTPUT STATISTICS =====
z1 mean : -0.20673143863677979
z1 std  : 1.6604434449618566e-07
z1 min  : -0.2067335844039917
z1 max  : -0.2067292332649231

z2 mean : 0.8279523849487305
z2 std  : 1.0366153717041016
z2 min  : -1.3819949626922607
z2 max  : 1.9819889068603516


In [94]:
model_path = "/content/drive/MyDrive/SaRLog/results/siamese_100k_model_corrected.pt"

torch.save(
    model_100k.state_dict(),
    model_path
)

print("Model saved successfully:")
print(model_path)

Model saved successfully:
/content/drive/MyDrive/SaRLog/results/siamese_100k_model_corrected.pt


okay now we are done training now we are gonna test it and see how the results are

In [95]:
# Load test embeddings and labels

test_embeddings_100k = torch.load(
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_test_embeddings.pt"
)

test_labels_100k = torch.load(
    "/content/drive/MyDrive/SaRLog/embeddings/bgl_100k_test_labels.pt"
)

print("Test embeddings:", test_embeddings_100k.shape)
print("Test labels:", test_labels_100k.shape)

print("Normal:", (test_labels_100k == 0).sum().item())
print("Anomaly:", (test_labels_100k == 1).sum().item())

Test embeddings: torch.Size([20000, 768])
Test labels: torch.Size([20000])
Normal: 18570
Anomaly: 1430


generate test representations

In [96]:
model_100k.eval()

test_embeddings_device = test_embeddings_100k.to(device)

with torch.no_grad():
    test_representations = model_100k.encoder(
        test_embeddings_device
    )

test_representations = test_representations.cpu()

print("Test representations:", test_representations.shape)
print("Mean:", test_representations.mean().item())
print("Std :", test_representations.std().item())
print("Min :", test_representations.min().item())
print("Max :", test_representations.max().item())

Test representations: torch.Size([20000, 1])
Mean: -0.05893728509545326
Std : 0.5332762002944946
Min : -0.28181344270706177
Max : 1.9819886684417725


In [97]:
# Generate representations for the training embeddings

model_100k.eval()

with torch.no_grad():
    train_representations = model_100k.encoder(
        train_embeddings_100k.to(device)
    )

train_representations = train_representations.cpu()

print("Training representations:", train_representations.shape)


Training representations: torch.Size([80000, 1])


In [98]:
# Find normal training samples
normal_train_indices = torch.where(
    train_labels_100k == 0
)[0]

# Use the first normal training representation as reference
reference_index = normal_train_indices[0]

normal_reference = train_representations[reference_index]

print("Reference index:", reference_index.item())
print("Reference representation:", normal_reference.item())

Reference index: 0
Reference representation: -0.2067314237356186


now we need to calc test distances

In [99]:
# Calculate distance from the normal reference
# for every test sample

distances_100k = torch.abs(
    test_representations - normal_reference
).squeeze(1)

print("Distances:", distances_100k.shape)

print("\nOverall distance statistics:")
print("Mean  :", distances_100k.mean().item())
print("Median:", distances_100k.median().item())
print("Min   :", distances_100k.min().item())
print("Max   :", distances_100k.max().item())

Distances: torch.Size([20000])

Overall distance statistics:
Mean  : 0.14780168235301971
Median: 7.450580596923828e-08
Min   : 0.0
Max   : 2.1887199878692627


In [100]:
normal_test_distances = distances_100k[test_labels_100k == 0]
anomaly_test_distances = distances_100k[test_labels_100k == 1]

print("===== NORMAL TEST LOGS =====")
print("Count :", len(normal_test_distances))
print("Mean  :", normal_test_distances.mean().item())
print("Median:", normal_test_distances.median().item())
print("Min   :", normal_test_distances.min().item())
print("Max   :", normal_test_distances.max().item())

print("\n===== ANOMALY TEST LOGS =====")
print("Count :", len(anomaly_test_distances))
print("Mean  :", anomaly_test_distances.mean().item())
print("Median:", anomaly_test_distances.median().item())
print("Min   :", anomaly_test_distances.min().item())
print("Max   :", anomaly_test_distances.max().item())

===== NORMAL TEST LOGS =====
Count : 18570
Mean  : 9.386126720301036e-08
Median: 7.450580596923828e-08
Min   : 0.0
Max   : 0.0002339482307434082

===== ANOMALY TEST LOGS =====
Count : 1430
Mean  : 2.067155361175537
Median: 2.016653537750244
Min   : 0.07508201897144318
Max   : 2.1887199878692627


In [101]:
# Paper-style anomaly detection
# distance > margin -> anomaly
# distance <= margin -> normal

threshold = 1.0

predictions_100k = (
    distances_100k > threshold
).long()

print("Predictions generated:", predictions_100k.shape)

print("\nPredicted normal:",
      (predictions_100k == 0).sum().item())

print("Predicted anomaly:",
      (predictions_100k == 1).sum().item())

Predictions generated: torch.Size([20000])

Predicted normal: 18571
Predicted anomaly: 1429


In [102]:
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

cm = confusion_matrix(
    test_labels_100k.numpy(),
    predictions_100k.numpy()
)

accuracy = accuracy_score(
    test_labels_100k.numpy(),
    predictions_100k.numpy()
)

precision = precision_score(
    test_labels_100k.numpy(),
    predictions_100k.numpy()
)

recall = recall_score(
    test_labels_100k.numpy(),
    predictions_100k.numpy()
)

f1 = f1_score(
    test_labels_100k.numpy(),
    predictions_100k.numpy()
)

print("===== CONFUSION MATRIX =====")
print(cm)

print("\n===== METRICS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

===== CONFUSION MATRIX =====
[[18570     0]
 [    1  1429]]

===== METRICS =====
Accuracy : 1.0000
Precision: 1.0000
Recall   : 0.9993
F1-score : 0.9997


In [103]:
import json

results = {
    "dataset": "BGL_100K",
    "train_samples": 80000,
    "test_samples": 20000,
    "normal_test": 18570,
    "anomaly_test": 1430,
    "architecture": "768-64-32-1",
    "epochs": 50,
    "batch_size": 32,
    "learning_rate": 0.001,
    "margin": 1.0,
    "threshold": 1.0,
    "accuracy": accuracy,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "confusion_matrix": cm.tolist()
}

results_path = "/content/drive/MyDrive/SaRLog/results/bgl_100k_results.json"

with open(results_path, "w") as f:
    json.dump(results, f, indent=4)

print("Results saved to:")
print(results_path)

Results saved to:
/content/drive/MyDrive/SaRLog/results/bgl_100k_results.json


In [105]:
print("Train columns:")
print(train_df.columns.tolist())

print("\nTest columns:")
print(test_df.columns.tolist())

print("\nTrain sample:")
display(train_df.head())

Train columns:
['label', 'message']

Test columns:
['label', 'message']

Train sample:


,label,message
0,0,generating core
1,0,generating core
2,0,generating core
3,0,generating core
4,0,generating core


In [107]:
train_messages = set(train_df["message"])
test_messages = set(test_df["message"])

overlap = train_messages.intersection(test_messages)

print("Unique train messages:", len(train_messages))
print("Unique test messages :", len(test_messages))
print("Messages appearing in both:", len(overlap))

print("\nExample overlapping messages:")
for msg in list(overlap)[:10]:
    print("-", msg)

Unique train messages: 314
Unique test messages : 166
Messages appearing in both: 125

Example overlapping messages:
- data address x
- ce sym at x cce mask x
- ce sym at x ad mask x
- fd eeeeeee b
- ce sym at x b fec mask x
- ce sym at x b ea mask x
- eeeeeee ffffffff ffffffff
- e ea ffffffff
- ce sym at x e be mask x
- ce sym at x cfdddc mask x


In [108]:
train_records = set(zip(train_df["label"], train_df["message"]))
test_records = set(zip(test_df["label"], test_df["message"]))

record_overlap = train_records.intersection(test_records)

print("Unique train records:", len(train_records))
print("Unique test records :", len(test_records))
print("Exact (label, message) overlap:", len(record_overlap))

print("\nExamples:")
for record in list(record_overlap)[:10]:
    print(record)


Unique train records: 314
Unique test records : 166
Exact (label, message) overlap: 125

Examples:
(0, 'ce sym at x bb mask x')
(0, 'general purpose registers')
(0, 'ce sym at x b f c mask x')
(0, 'problem state sup usr')
(0, 'e e feaa')
(0, 'ce sym at x b eec mask x')
(0, 'ce sym at x cb e mask x')
(0, 'ce sym at x ed mask x')
(0, 'ce sym at x b eee mask x')
(0, 'force load store alignment')
